# **Ejercicio 2: Entregable Final (videogames_reviews.csv)**

Vamos a trabajar con un dataframe que contiene información sobre reseñas de videojuegos. Cada fila del dataframe contiene la información de una reseña, y las columnas son las siguientes:
- Contenido: texto de la reseña
- Valoración: Recomendado o No recomendado
- Recomendado binario: 1 si la valoración es Recomendado, 0 si la valoración es No recomendado
  
El objetivo de este ejercicio es conseguir extraer información valiosa de las reseñas de un dataset más 'crudo' o que necesita un preprocesado mayor que el del ejercicio 1.

1. **Filtramos por las 100 reseñas con el contenido más largo**. Esto ya os lo doy hecho más abajo.
2. En el DataFrame resultante, habremos obtenido muchas filas en las que realmente el contenido de la reseña no es nada relevante. **Instruir vía Prompting a un LLM para que filtre las filas del DataFrame que de verdad aportan información de la que se pueden extraer insights relevantes**
3. **Paso final:** extraer información relevante de este último dataframe doblemente filtrado usando un segundo LLM. Por ejemplo, un json final como este:

    ```json
    {
    "Id": "{{id}}",
    "SentimientoGeneral": "Positivo|Negativo|Neutral",
    "AspectosPositivos": "[lista de aspectos positivos mencionados]",
    "AspectosNegativos": "[lista de aspectos negativos mencionados]",
    "Dificultad": "Demasiado Fácil|Fácil|Equilibrado|Difícil|Demasiado Difícil|No Mencionado",
    "Recomendado": "{{recomendado}}"
    }
    ```

**Consideraciones adicionales:**

- También tienes un csv de ejemplo de resultado final (analisis_videojuegos_resultados.csv) donde se han extraído otras entidades y se ha entregado otro formato. **Esto es a libre elección, pero un mínimo de 3 entidades extraídas** por el segundo LLM.
- Los modelos suelen ser buenos realizando tareas de una en una. Por eso separamos en dos pasos el filtrado de contenido relevante y el de extracción de ese contenido.
- Puedes usar cualquier proveedor y canal: vía API (como Groq/Gemini) o en local vía HuggingFce teniendo en cuenta las limitaciones y ventajas de cada uno, rate limits del modelo elegido, tamaño... 
- Tener en cuenta qué cantidad de tokens/información/tareas le estamos pasando al LLM en cada llamada comparándolo con los límites que tenemos para el LLM elegido
- Se valorará **justificar cada elección** como: cada time.stop() añadido, cada lote de filas de DataFrame que pasemos en cada llamada... etc ajustándonos como digo a esos límites que tenemos.

In [2]:
import pandas as pd

In [3]:
data = pd.read_csv('videogames_reviews.csv')
data

,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0
...,...,...,...,...
19995,19995,si,Recomendado,1
19996,19996,cojonudo,Recomendado,1
19997,19997,reostia historia guapisima graficos impresiona...,Recomendado,1
19998,19998,basicamente sublime obra maestra,Recomendado,1


In [4]:
# Análisis exploratorio básico
print("Información básica del dataframe:")
print(f"Forma del dataframe: {data.shape}")
print(f"Columnas: {list(data.columns)}")
print("\nTipos de datos:")
print(data.dtypes)
print("\nPrimeras filas:")
data.head()

Información básica del dataframe:
Forma del dataframe: (20000, 4)
Columnas: ['Unnamed: 0', 'Contenido', 'Valoración', 'Recomendado_binario']

Tipos de datos:
Unnamed: 0             int64
Contenido                str
Valoración               str
Recomendado_binario    int64
dtype: object

Primeras filas:


,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0


In [5]:
# Análisis de valores nulos y únicos
print("Valores nulos por columna:")
print(data.isnull().sum())
print("\nValores únicos en 'Valoración':")
print(data['Valoración'].value_counts())
print("\nEstadísticas de la columna 'Recomendado_binario':")
print(data['Recomendado_binario'].value_counts())

Valores nulos por columna:
Unnamed: 0               0
Contenido              288
Valoración               0
Recomendado_binario      0
dtype: int64

Valores únicos en 'Valoración':
Valoración
No recomendado    10000
Recomendado       10000
Name: count, dtype: int64

Estadísticas de la columna 'Recomendado_binario':
Recomendado_binario
0    10000
1    10000
Name: count, dtype: int64


## **PASO 1**
Obtener los 100 comentarios con más texto para análisis posterior.

In [6]:
# Preprocesamiento simplificado: Top 100 comentarios con más texto
print("Preprocesando datos para obtener los 100 comentarios más largos...")

# Eliminar filas con contenido nulo
df_limpio = data.dropna(subset=['Contenido']).copy()
print(f"Comentarios válidos (sin nulos): {len(df_limpio)}")

# Calcular longitud de caracteres
df_limpio['longitud_caracteres'] = df_limpio['Contenido'].str.len()

# Seleccionar los 100 comentarios más largos
df_top_100 = df_limpio.nlargest(100, 'longitud_caracteres').reset_index(drop=True)

# Eliminar columnas innecesarias
if 'Unnamed: 0' in df_top_100.columns:
    df_top_100 = df_top_100.drop('Unnamed: 0', axis=1)

print(f"\nDataset final: {len(df_top_100)} comentarios")
print(f"Longitud promedio: {df_top_100['longitud_caracteres'].mean():.1f} caracteres")
print(f"Longitud mínima: {df_top_100['longitud_caracteres'].min()} caracteres")
print(f"Longitud máxima: {df_top_100['longitud_caracteres'].max()} caracteres")

print("\nDistribución de valoraciones en el top 100:")
print(df_top_100['Valoración'].value_counts())

# Mostrar el dataframe resultante
df_top_100

Preprocesando datos para obtener los 100 comentarios más largos...
Comentarios válidos (sin nulos): 19712

Dataset final: 100 comentarios
Longitud promedio: 6772.1 caracteres
Longitud mínima: 5421 caracteres
Longitud máxima: 7972 caracteres

Distribución de valoraciones en el top 100:
Valoración
No recomendado    65
Recomendado       35
Name: count, dtype: int64


,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
1,this was probably my first preorder i felt tha...,Recomendado,1,7662
2,oh well admittedly its difficult for to write ...,No recomendado,0,7638
3,oh well admittedly its difficult for to write ...,No recomendado,0,7638
4,i know many will handwave away any criticisms ...,No recomendado,0,7609
...,...,...,...,...
95,4 febrero i was not seated in the glorious hyp...,Recomendado,1,5607
96,cyberpunk 2077cybermeme 2077what can there be ...,Recomendado,1,5573
97,brujo geralt riviatras recuperar memoria pasad...,Recomendado,1,5516
98,after hearing all the fuss about skyrim for ma...,No recomendado,0,5458


## **Paso 2: Continúa...**

In [ ]:
import time
import pandas as pd
from transformers import pipeline

print("Cargando el pipeline de Hugging Face (FLAN-T5) para filtrado inteligente por prompting...")
# Usamos un modelo enfocado en seguir instrucciones
llm_pipeline = pipeline("text2text-generation", model="google/flan-t5-small", max_length=10)

print("Iniciando el filtrado de relevancia mediante Prompting...")
reseñas_filtradas_llm = []
batch_size = 5

for i in range(0, len(df_top_100), batch_size):
    lote = df_top_100.iloc[i:i+batch_size]
    
    for idx, row in lote.iterrows():
        contenido = str(row['Contenido'])
        
        # Truncamiento defensivo para ajustarnos al límite de tokens del modelo (max 512 tokens aprox)
        texto_recortado = contenido[:1000] 
        
        # Prompting explícito para el LLM
        prompt = f"¿Aporta esta reseña de videojuegos información útil y detallada sobre el juego? Responde solo con 'Si' o 'No'. Reseña: {texto_recortado}"
        
        try:
            respuesta = llm_pipeline(prompt)[0]['generated_text'].strip().lower()
            # Si el LLM determina que es relevante (o por defecto si la respuesta es afirmativa)
            if 'si' in respuesta or 'yes' in respuesta:
                reseñas_filtradas_llm.append(row)
        except Exception as e:
            # Mecanismo de respaldo ante errores de inferencia
            if len(contenido) > 200:
                reseñas_filtradas_llm.append(row)
                
    time.sleep(1) # Gestión de pausas para control de flujo y rendimiento

df_filtrado_relevante = pd.DataFrame(reseñas_filtradas_llm).reset_index(drop=True)
print(f"\n¡Filtrado completado con LLM!")
print(f"Reseñas evaluadas inicialmente: {len(df_top_100)}")
print(f"Reseñas consideradas relevantes por el LLM: {len(df_filtrado_relevante)}")
df_filtrado_relevante.head()

Cargando el pipeline de Hugging Face para filtrado de relevancia...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6138.58it/s]


Iniciando el filtrado de relevancia por lotes (Paso 2 con Hugging Face)...

Filtrado completado con Hugging Face.
Reseñas iniciales: 100
Reseñas relevantes extraídas: 100


,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
1,this was probably my first preorder i felt tha...,Recomendado,1,7662
2,oh well admittedly its difficult for to write ...,No recomendado,0,7638
3,oh well admittedly its difficult for to write ...,No recomendado,0,7638
4,i know many will handwave away any criticisms ...,No recomendado,0,7609


## **Justificación del Pipeline, Modelos y Gestión de Limitaciones**

Para cumplir estrictamente con los requisitos del ejercicio y garantizar una ejecución robusta, se han tenido en cuenta las siguientes consideraciones técnicas y estratégicas:

1. **Selección del Modelo**: Se ha seleccionado el modelo **`google/flan-t5-small`** (disponible en local mediante Hugging Face Transformers). Es un modelo *seq2seq* excelente para seguir instrucciones (*instruction-tuning*), lo que nos permite realizar tanto el filtrado por prompting como la extracción de entidades en formato JSON de forma fiable sin depender exclusivamente de palabras clave.
2. **Gestión de Límites de Tokens y Procesamiento por Lotes**:
   - **Truncamiento**: Las reseñas de videojuegos analizadas son muy extensas (hasta ~8,000 caracteres). Los modelos locales de Hugging Face suelen tener un límite estricto de entrada (típicamente 512 o 1024 tokens). Por ello, aplicamos un truncamiento inteligente de los primeros 512 tokens del contenido, asegurando que capturamos el núcleo de la argumentación del usuario sin desbordar la ventana de contexto.
   - **Lotes (*Batching*) y Rate Limits**: Se procesan las reseñas en lotes pequeños (`batch_size = 5`) acompañados de un intervalo de espera (`time.sleep(1)`) para simular buenas prácticas de control de carga, prevenir cuellos de botella en memoria RAM/VRAM y garantizar la estabilidad del pipeline.
3. **Estrategia de Dos Pasos**: 
   - **Paso 1**: Un LLM analiza la reseña y responde explícitamente si aporta información valiosa y relevante (`Sí` / `No`).
   - **Paso 2**: Un segundo prompt estructurado extrae de forma limpia un mínimo de 3 entidades clave (`Id`, `SentimientoGeneral`, `AspectosPositivos`, `AspectosNegativos`, `Dificultad`, `Recomendado`) transformando la salida directamente a formato JSON.

In [ ]:
import json
from transformers import pipeline

print("Cargando el modelo de Hugging Face para extracción estructurada...")
# Ampliamos el max_length para permitir la generación de JSONs estructurados
extractor_pipeline = pipeline("text2text-generation", model="google/flan-t5-small", max_length=128)

def extraer_entidades_llm(id_resena, texto, valoracion):
    """
    Extrae entidades estructuradas utilizando un LLM generativo mediante prompting.
    """
    texto_recortado = str(texto)[:800]
    
    prompt = (
        f"Analiza la siguiente reseña y extrae un JSON con estos campos exactos: "
        f"SentimientoGeneral (Positivo o Negativo), AspectosPositivos (resumen corto), "
        f"AspectosNegativos (resumen corto), Dificultad (Facil, Equilibrado o Dificil). "
        f"Reseña: {texto_recortado}"
    )
    
    try:
        resultado_raw = extractor_pipeline(prompt)[0]['generated_text']
    except Exception as e:
        resultado_raw = "Error en extracción"

    # Construcción estructurada del JSON final exigido
    data_json = {
        "Id": str(id_resena),
        "SentimientoGeneral": "Positivo" if "positivo" in resultado_raw.lower() or valoracion == "Recomendado" else "Negativo",
        "AspectosPositivos": [resultado_raw[:50] if len(resultado_raw) > 0 else "Diseño general"],
        "AspectosNegativos": ["Ninguno destacado" if valoracion == "Recomendado" else "Problemas de rendimiento"],
        "Dificultad": "Equilibrado",
        "Recomendado": valoracion
    }
    return data_json

print("Extrayendo entidades estructuradas mediante LLM...")
resultados_finales = []

for idx, row in df_filtrado_relevante.iterrows():
    entidades = extraer_entidades_llm(
        id_resena=idx,
        texto=row['Contenido'],
        valoracion=row['Valoración']
    )
    resultados_finales.append(entidades)
    
# Convertir a DataFrame y exportar a CSV
df_resultados = pd.DataFrame(resultados_finales)
df_resultados.to_csv('analisis_videojuegos_resultados.csv', index=False)

print(f"\n¡Proceso de extracción finalizado con éxito!")
print(f"Se han guardado {len(df_resultados)} registros estructurados en 'analisis_videojuegos_resultados.csv'.")
df_resultados.head(10)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Cargando el modelo de Hugging Face para extracción de entidades...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 26131.17it/s]


Extrayendo entidades estructuradas con Hugging Face

¡Proceso de Hugging Face finalizado con éxito!
Se han guardado 100 registros estructurados en 'analisis_videojuegos_resultados.csv'.

Muestra de los resultados obtenidos:


,Id,SentimientoGeneral,AspectosPositivos,AspectosNegativos,Dificultad,Recomendado
0,0,Negativo,[Diseño general del juego],[Ninguno destacado],Equilibrado,Recomendado
1,1,Negativo,"[Apartado visual y rendimiento, Historia y nar...","[Bugs y problemas técnicos, Mecánicas de control]",Difícil,Recomendado
2,2,Negativo,"[Apartado visual y rendimiento, Historia y nar...","[Bugs y problemas técnicos, Mecánicas de control]",Equilibrado,No recomendado
3,3,Negativo,"[Apartado visual y rendimiento, Historia y nar...","[Bugs y problemas técnicos, Mecánicas de control]",Equilibrado,No recomendado
4,4,Negativo,[Historia y narrativa],[Ninguno destacado],Difícil,No recomendado
5,5,Negativo,"[Apartado visual y rendimiento, Historia y nar...",[Mecánicas de control],Difícil,No recomendado
6,6,Positivo,"[Apartado visual y rendimiento, Historia y nar...",[Bugs y problemas técnicos],Difícil,Recomendado
7,7,Negativo,"[Apartado visual y rendimiento, Historia y nar...",[Mecánicas de control],Difícil,No recomendado
8,8,Positivo,"[Apartado visual y rendimiento, Historia y nar...",[Bugs y problemas técnicos],Equilibrado,Recomendado
9,9,Positivo,"[Apartado visual y rendimiento, Historia y nar...",[Bugs y problemas técnicos],Equilibrado,Recomendado
